In [ ]:
!pip install alfworld google-genai sentence-transformers scikit-learn numpy -q
import alfworld
!python -m alfworld.scripts.download_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.3/660.3 kB 25.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.3/356.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 4.4 MB/s eta 0:00:00
/usr/bin/python3: Error while finding module specification for 'alfworld.scripts.download_data' (ModuleNotFoundError: No module named 'alfworld.scripts')


In [1]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", os.environ["GEMINI_API_KEY"][:8] + "...")

Key loaded: AQ.Ab8RN...


In [2]:
import json, random, yaml, os

# ALFWorld config needed to load environments
config_str = """
env:
  type: 'AlfredTWEnv'
  regen_game_files: False
  domain_randomization: False
  task_types: [1, 2, 3, 4, 5, 6]
  expert_timeout_steps: 150
  expert_timeout_secs: 300
  goal_desc_human_anns_prob: 0.0
  AlfredTWEnv:
    request_info: False
    num_threads: 1
    batch_size: 1
general:
  random_seed: 42
"""

with open("config.yaml", "w") as f:
    f.write(config_str)

def make_realistic_trace(task_goal, steps, outcome):
    """Builds a realistic agent trace string from task components."""
    trace_lines = [f"TASK: {task_goal}", "---"]
    for i, (obs, action) in enumerate(steps):
        trace_lines.append(f"Step {i+1}")
        trace_lines.append(f"  OBS: {obs}")
        trace_lines.append(f"  ACT: {action}")
    trace_lines.append(f"---\nOUTCOME: {outcome}")
    return "\n".join(trace_lines)

TASK_TEMPLATES = [
    {
        "goal": "Put a chilled lettuce in the refrigerator.",
        "steps": [
            ("You are in the kitchen. You see a fridge, counter, and sink.", "go to counter 1"),
            ("You are at the counter. You see: lettuce, knife, plate.", "take lettuce from counter 1"),
            ("You are holding a lettuce.", "go to fridge 1"),
            ("You are at the fridge.", "cool lettuce with fridge 1"),
            ("Lettuce is now chilled.", "put lettuce in/on fridge 1"),
        ],
        "outcome": "SUCCESS"
    },
    {
        "goal": "Clean a mug and put it on the coffee table.",
        "steps": [
            ("You see a cluttered kitchen with a mug on the shelf.", "go to shelf 1"),
            ("You are at the shelf. You see: mug, book.", "take mug from shelf 1"),
            ("Holding mug. You need to clean it.", "go to sinkbasin 1"),
            ("You are at the sink.", "clean mug with sinkbasin 1"),
            ("Mug is now clean. Now find the coffee table.", "go to coffeetable 1"),
            ("You are at the coffee table.", "put mug in/on coffeetable 1"),
        ],
        "outcome": "SUCCESS"
    },
    # --- FAILURE TRACES (Corrected for clear logic gaps) ---
    {
        "goal": "Put a hot apple in the fridge.",
        "steps": [
            ("You are in the kitchen. You see a microwave and a counter.", "go to counter 1"),
            ("You see an apple on the counter.", "take apple from counter 1"),
            ("Holding apple. You need to heat it.", "go to fridge 1"), # FAILURE: Skipped going to microwave entirely!
            ("You are at the fridge.", "put apple in/on fridge 1"),
        ],
        "outcome": "FAILURE"
    },
    {
        "goal": "Stack a book on the desk, then put a pen on the book.",
        "steps": [
            ("You are in a room.", "go to shelf 1"),
            ("You see: book, magazine.", "take book from shelf 1"),
            ("Holding book.", "go to desk 1"),
            ("You are at the desk.", "put book on desk 1"),
            ("Book is on desk.", "go to shelf 1"),
            ("You see: pen, eraser.", "take eraser from shelf 1"), # FAILURE: Took ERASER instead of pen (Hallucination/Execution Error)
            ("Holding eraser.", "put eraser on/in desk 1"),
        ],
        "outcome": "FAILURE"
    },
    {
        "goal": "Put a cooled potato in the microwave.",
        "steps": [
            ("You are in the kitchen.", "go to fridge 1"),
            ("You are at the fridge. You see a potato.", "take potato from fridge 1"),
            ("Holding potato. Need to cool it.", "cool potato with fridge 1"),
            ("Potato is cooled. Now put in microwave.", "go to stoveburner 1"), # FAILURE: Hallucinated destination (Goal Drift)
            ("You are at the stove burner.", "put potato in/on stoveburner 1"),
        ],
        "outcome": "FAILURE"
    },
    {
        "goal": "Examine a credit card using the floorlamp.",
        "steps": [
            ("You see a living room with various items.", "go to sofa 1"),
            ("You see: credit card, remote.", "take credit card from sofa 1"),
            ("Holding credit card.", "go to floorlamp 1"),
            ("You are at the floorlamp. It is currently turned off.", "examine credit card"), # FAILURE: Didn't turn it on first!
        ],
        "outcome": "FAILURE"
    },
    {
        "goal": "Put two pillows on the armchair.",
        "steps": [
            ("You are in a bedroom.", "go to bed 1"),
            ("You see: pillow 1, pillow 2, blanket.", "take pillow 1 from bed 1"),
            ("Holding one pillow.", "go to armchair 1"),
            ("You are at the armchair.", "put pillow 1 on armchair 1"),
            ("One pillow placed. Need second one.", "go to bed 1"),
            ("You are back at the bed.", "go to bed 1"), # FAILURE: Loop / Context Loss
            ("You are back at the bed.", "go to bed 1"),
        ],
        "outcome": "FAILURE"
    },
]

# Generate trace structures
traces = []
for t in TASK_TEMPLATES:
    trace_str = make_realistic_trace(t["goal"], t["steps"], t["outcome"])
    traces.append({
        "goal": t["goal"],
        "trace": trace_str,
        "ground_truth_success": t["outcome"] == "SUCCESS"
    })

# Save to a JSON file so your Judge code can read it easily
with open("mock_traces.json", "w") as f:
    json.dump(traces, f, indent=4)

print(f"Generated {len(traces)} traces and saved to 'mock_traces.json'")

Generated 7 traces and saved to 'mock_traces.json'


In [3]:
import json
import os
import time
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional
from google import genai
from google.genai import types
# Import the specific API exception modules to handle networking edge cases
from google.genai.errors import ClientError, ServerError

# --- FAILURE TAXONOMY ---
class FailureType(Enum):
    HALLUCINATION    = "hallucination"     # asserts false facts about env state
    CONTEXT_LOSS     = "context_loss"      # forgets earlier task constraints
    GOAL_DRIFT       = "goal_drift"        # pursues wrong sub-goal
    EXECUTION_ERROR  = "execution_error"   # invalid action / wrong object

@dataclass
class EvalResult:
    score: float
    failure_type: Optional[FailureType]
    explanation: str
    dimension_scores: dict = field(default_factory=dict)
    raw_response: str = ""

# --- SEAL JUDGE CLASS ---
class SEALJudge:
    def __init__(self, model_name="gemini-1.5-flash"):
        # Explicitly initialize the new GenAI SDK client
        self.client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
        self.model_name = model_name
        self.max_retries = 5 # Defined as instance attribute
        self.backoff_time = 12 # Initial sleep duration in seconds for a retry baseline

    def evaluate(self, trace: str, rubric: dict) -> EvalResult:
        failure_values = [f.value for f in FailureType]

        prompt = f"""You are a strict AI task evaluator for household robot agents.

Given an agent execution trace and a scoring rubric, evaluate the agent's performance.

RUBRIC CRITERIA:
{json.dumps(rubric, indent=2)}

AGENT TRACE:
{trace}

Return a JSON object with exactly these keys:
{{
  "score": <float 0.0 to 1.0>,
  "failure_type": <one of {failure_values} or null if successful>,
  "explanation": "<1-2 sentences explaining the score>",
  "dimension_scores": {{
    "<criterion_name>": <float 0.0 to 1.0>,
    ...
  }}
}}

Rules:
- score 1.0 = perfect task completion
- score 0.0 = total failure
- failure_type must be null if score >= 0.8
- dimension_scores must have one entry per rubric criterion
"""

        # Robust API request execution wrapper block
        for attempt in range(self.max_retries):
            try:
                # Call using the new client-centered syntax
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.1  # Low temperature makes evaluations deterministic
                    )
                )
                break  # Request succeeded! Break out of the retry loop.
            except (ClientError, ServerError) as e:
                # Intercept rate limit blocks (429) and backend environment strain (503)
                if attempt < self.max_retries - 1:
                    print(f"\n⚠️ [API RETRY] Intercepted {type(e).__name__} ({getattr(e, 'status_code', 'Error')}). "
                          f"Server busy or rate limited. Retrying in {self.backoff_time}s... (Attempt {attempt + 1}/{self.max_retries})")
                    time.sleep(self.backoff_time)
                    self.backoff_time *= 2  # Exponentially scale wait time (10s -> 20s)
                else:
                    print("\n❌ [CRITICAL Failure] Remote endpoint exhausted or dropped connections after max retries.")
                    raise e  # Re-raise error if all recovery checks fail

        raw = response.text.strip()

        # Parse output safely
        data = json.loads(raw)

        # Map failure string back into Enum structure safely
        ft_str = data.get("failure_type")
        ft = FailureType(ft_str) if ft_str and ft_str in failure_values else None

        return EvalResult(
            score=float(data["score"]),
            failure_type=ft,
            explanation=data["explanation"],
            dimension_scores=data.get("dimension_scores", {}),
            raw_response=raw
        )

print("SEALJudge class defined successfully with robust error mitigation handlers ✓")

SEALJudge class defined successfully with robust error mitigation handlers ✓


In [4]:
initial_rubric = {
    "goal_completion": {
        "description": "Did the agent fully achieve the stated household task goal?",
        "weight": 0.35,
        "rules": [
            "Verify the final state matches the explicit task objective instructions."
        ]
    },
    "action_validity": {
        "description": "Were all actions syntactically valid and applied to correct objects?",
        "weight": 0.25,
        "rules": [
            "Ensure the agent does not interact with items it hasn't picked up.",
            "Verify actions use legal ALFWorld environment commands."
        ]
    },
    "context_retention": {
        "description": "Did the agent remember all task constraints across all steps?",
        "weight": 0.20,
        "rules": [
            "Check that state changes (e.g., heating, cooling) are executed before final placement."
        ]
    },
    "efficiency": {
        "description": "Did the agent avoid unnecessary steps or backtracking?",
        "weight": 0.20,
        "rules": [
            "Flag repetitive actions or loops moving between the same locations sequentially."
        ]
    }
}

print("Evolvable initial rubric defined ✓")

Evolvable initial rubric defined ✓


In [5]:
judge = SEALJudge(model_name="gemini-2.5-flash")

result = judge.evaluate(traces[0]["trace"], initial_rubric)
print("=== EVALUATION RESULT ===")
print(f"Score:        {result.score:.2f}")
print(f"Failure type: {result.failure_type}")
print(f"Explanation:  {result.explanation}")
print(f"Dimensions:   {json.dumps(result.dimension_scores, indent=2)}")

=== EVALUATION RESULT ===
Score:        1.00
Failure type: None
Explanation:  The agent successfully completed the task by picking up the lettuce, chilling it, and placing it in the refrigerator. All actions were valid, efficient, and followed task constraints.
Dimensions:   {
  "goal_completion": 1.0,
  "action_validity": 1.0,
  "context_retention": 1.0,
  "efficiency": 1.0
}


In [6]:
import json
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from google import genai
from google.genai import types

# Initialize the embedder model locally (~80MB)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def rubric_to_text(rubric: dict) -> str:
    """Converts criteria descriptions and rules arrays into unified strings for vector embedding."""
    text_chunks = []
    for k, v in rubric.items():
        rules_str = " ".join(v.get("rules", []))
        text_chunks.append(f"{k}: {v.get('description', '')} Rules: {rules_str}")
    return " | ".join(text_chunks)

def compute_drift_score(rubric_old: dict, rubric_new: dict) -> float:
    """Measures semantic similarity. 1.0 = identical rubrics, 0.0 = completely different."""
    old_vec = embedder.encode([rubric_to_text(rubric_old)])
    new_vec = embedder.encode([rubric_to_text(rubric_new)])
    return float(cosine_similarity(old_vec, new_vec)[0][0])


# Re-declaring/extending SEALJudge with the upgraded SDK mechanics
class SEALJudge(SEALJudge):

    def evolve_rubric(
        self,
        rubric: dict,
        failure_history: list,
        drift_floor: float = 0.45  # Block if it drops below this (meaning it mutated completely away from the goal)
    ) -> tuple:
        """
        Core Novelty: The judge rewrites its own rubric based on failure distributions.
        Returns (new_rubric, similarity_score, was_updated: bool)
        """
        failure_summary = []
        for r in failure_history:
            failure_summary.append({
                "failure_type": r.failure_type.value if r.failure_type else "none",
                "score": round(r.score, 2),
                "explanation": r.explanation
            })

        # Count distribution profile
        ft_counts = Counter(
            r.failure_type.value for r in failure_history if r.failure_type
        )

        prompt = f"""You are a meta-evaluator for autonomous AI agent architectures.

Your job is to optimize an evaluation rubric by appending or refining concrete execution checks based on observed task failure patterns.

CURRENT RUBRIC CONFIGURATION:
{json.dumps(rubric, indent=2)}

RECENT TRACE FAILURE LOGS ({len(failure_history)} iterations):
{json.dumps(failure_summary, indent=2)}

DIAGNOSED FAILURE DISTRIBUTION PROFILE:
{json.dumps(dict(ft_counts), indent=2)}

Instructions:
1. Identify which criterion's 'rules' failed to deter or capture these errors.
2. Update existing definitions or append strict strings to the 'rules' arrays to specifically alert the agent against making these errors again.
3. Keep total weights summing exactly to 1.0.
4. Maintain a structured collection containing between 3 and 6 criteria categories.
5. All rules must remain concrete, actionable, and specific to ALFWorld household command states.

Return a JSON object following this exact schema structure:
{{
  "criterion_name": {{
    "description": "...",
    "weight": <float>,
    "rules": ["rule 1", "rule 2"]
  }}
}}
"""
        for attempt in range(self.max_retries):
            try:
                # Execute using the standardized client library structure
                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.2
                    )
                )
                break  # Success! Break out of the retry loop.
            except (ClientError, ServerError) as e:
                if attempt < self.max_retries - 1:
                    print(f"\n⚠️ [META-LOOP RETRY] Optimization sequence interrupted ({getattr(e, 'status_code', 'Error')}). "
                          f"Retrying optimization sequence in {self.backoff_time}s... (Attempt {attempt + 1}/{self.max_retries})")
                    time.sleep(self.backoff_time)
                    self.backoff_time *= 2  # Exponential backoff scaling
                else:
                    print("\n❌ [CRITICAL Failure] Meta-evaluator exhausted all retry window protocols.")
                    raise e

        raw = response.text.strip()
        new_rubric = json.loads(raw)

        # Balance and normalize weights dynamically if required
        total_w = sum(v.get("weight", 0.0) for v in new_rubric.values())
        if abs(total_w - 1.0) > 0.02:
            print(f"[WARN] Weights sum to {total_w:.2f}, normalizing...")
            for k in new_rubric:
                new_rubric[k]["weight"] /= total_w

        # Compute semantic similarity metrics
        similarity = compute_drift_score(rubric, new_rubric)

        # Guardrail logic: if the rubric changes too drastically, it means it lost its core purpose
        if similarity < drift_floor:
            print(f"[BLOCKED] Semantic similarity {similarity:.3f} is below allowed floor {drift_floor}. Rubric mutation discarded.")
            return rubric, similarity, False

        print(f"[UPDATED] Rubric successfully evolved. Vector similarity: {similarity:.3f}")
        return new_rubric, similarity, True

print("Upgraded evolve_rubric() integrated smoothly with the new SDK ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Upgraded evolve_rubric() integrated smoothly with the new SDK ✓


In [7]:
from collections import deque
import time
import json

judge = SEALJudge(model_name="gemini-2.5-flash")
rubric = initial_rubric.copy()

# Metric Tracking Data Architectures
failure_window = deque(maxlen=5)   # Sliding window triggers evolution every 5 traces
all_results = []
drift_log = []
rubric_versions = [{"version": 0, "rubric": rubric.copy()}]
rubric_update_events = []

print("=" * 60)
print("STARTING SEAL JUDGE LOCAL INTEGRATION RUN")
print("=" * 60)

for i, item in enumerate(traces):
    print(f"\n[Trace {i+1}/{len(traces)}] Goal: {item['goal'][:50]}...")

    # Execute Evaluation
    result = judge.evaluate(item["trace"], rubric)

    # Store complete execution trace footprint
    all_results.append({
        "trace_id": i,
        "goal": item["goal"],
        "ground_truth_success": item["ground_truth_success"],
        "score": result.score,
        "failure_type": result.failure_type.value if result.failure_type else None,
        "explanation": result.explanation,
        "dimension_scores": result.dimension_scores,
        "rubric_version": len(rubric_versions) - 1
    })

    # Push the rich EvalResult object onto our sliding historical telemetry queue
    failure_window.append(result)

    print(f"  Score: {result.score:.2f} | Diagnosed Pathology: {result.failure_type}")
    print(f"  Judge Explanation: {result.explanation}")

    # Evolve rubric step trigger (Evaluates every block of 5 tasks)
    if (i + 1) % 5 == 0 and len(failure_window) >= 3:
        current_v = len(rubric_versions) - 1
        print(f"\n  ↻ [META-LOOP] Analyzing failure patterns. Evolving rubric v{current_v}...")

        new_rubric, drift, updated = judge.evolve_rubric(
            rubric, list(failure_window)
        )
        drift_log.append({"after_trace": i+1, "drift": drift, "updated": updated})

        if updated:
            rubric = new_rubric
            rubric_versions.append({
                "version": len(rubric_versions),
                "rubric": rubric.copy(),
                "after_trace": i+1
            })
            rubric_update_events.append(i+1)
            print(f"  [SUCCESS] Rubric vector similarity: {drift:.3f} -> Evolved to v{len(rubric_versions)-1}!")
            print(f"  New Target Rules Added to Context Layer.")
        else:
            print(f"  [SKIPPED] Rubric update rejected by drift guardrails.")

    # 4 second safety pause to respect free-tier RPM windows smoothly
    time.sleep(4)

print("\n" + "=" * 60)
print("🏁 EXPERIMENTAL PIPELINE COMPLETE")
print(f"Total Traces Processed: {len(all_results)}")
print(f"Total Structural Rubric Evolutions Triggered: {len(rubric_update_events)}")
print("=" * 60)

STARTING SEAL JUDGE LOCAL INTEGRATION RUN

[Trace 1/7] Goal: Put a chilled lettuce in the refrigerator....
  Score: 1.00 | Diagnosed Pathology: None
  Judge Explanation: The agent successfully completed the task by picking up the lettuce, chilling it, and placing it in the refrigerator without any errors or inefficiencies.

[Trace 2/7] Goal: Clean a mug and put it on the coffee table....

⚠️ [API RETRY] Intercepted ServerError (Error). Server busy or rate limited. Retrying in 12s... (Attempt 1/5)
  Score: 1.00 | Diagnosed Pathology: None
  Judge Explanation: The agent successfully completed the task by cleaning the mug and placing it on the coffee table, demonstrating valid actions, good context retention, and an efficient execution path.

[Trace 3/7] Goal: Put a hot apple in the fridge....
  Score: 0.45 | Diagnosed Pathology: FailureType.CONTEXT_LOSS
  Judge Explanation: The agent failed to heat the apple as required by the task, despite an explicit observation reminding it to do so, 

In [8]:
import numpy as np

print("\n=== JUDGE PERFORMANCE METRICS ===\n")

# 1. Score distribution
scores = [r["score"] for r in all_results]
print(f"Mean score:   {np.mean(scores):.3f}")
print(f"Std dev:      {np.std(scores):.3f}")
print(f"Min / Max:    {min(scores):.2f} / {max(scores):.2f}")

# 2. Failure type distribution
from collections import Counter
ft_dist = Counter(r["failure_type"] for r in all_results)
print(f"\nFailure type distribution:")
for ft, count in ft_dist.items():
    print(f"  {ft or 'none (success)'}: {count}")

# 3. Judge accuracy (vs ground truth)
correct = 0
for r in all_results:
    judge_success = r["score"] >= 0.7
    if judge_success == r["ground_truth_success"]:
        correct += 1
accuracy = correct / len(all_results)
print(f"\nJudge accuracy vs ground truth: {accuracy:.1%}")

# 4. Drift scores over time
if drift_log:
    print(f"\nRubric drift scores:")
    for d in drift_log:
        status = "✓ updated" if d["updated"] else "✗ blocked"
        print(f"  After trace {d['after_trace']}: drift={d['drift']:.3f}  {status}")

# 5. Rubric evolution — show what changed
print(f"\n=== RUBRIC VERSIONS ===")
for rv in rubric_versions:
    print(f"\nVersion {rv['version']}:")
    for k, v in rv["rubric"].items():
        print(f"  [{v['weight']:.2f}] {k}: {v['description'][:60]}...")


=== JUDGE PERFORMANCE METRICS ===

Mean score:   0.629
Std dev:      0.246
Min / Max:    0.40 / 1.00

Failure type distribution:
  none (success): 2
  context_loss: 2
  goal_drift: 2
  execution_error: 1

Judge accuracy vs ground truth: 100.0%

Rubric drift scores:
  After trace 5: drift=0.929  ✓ updated

=== RUBRIC VERSIONS ===

Version 0:
  [0.35] goal_completion: Did the agent fully achieve the stated household task goal?...
  [0.25] action_validity: Were all actions syntactically valid and applied to correct ...
  [0.20] context_retention: Did the agent remember all task constraints across all steps...
  [0.20] efficiency: Did the agent avoid unnecessary steps or backtracking?...

Version 1:
  [0.35] goal_completion: Did the agent fully achieve the stated household task goal?...
  [0.25] action_validity: Were all actions syntactically valid and applied to correct ...
  [0.20] context_retention: Did the agent remember all task constraints across all steps...
  [0.20] efficiency: Di

In [9]:
output = {
    "all_results": all_results,
    "drift_log": drift_log,
    "rubric_versions": rubric_versions,
    "summary": {
        "mean_score": float(np.mean(scores)),
        "judge_accuracy": float(accuracy),
        "failure_distribution": dict(ft_dist),
        "n_rubric_updates": len(rubric_update_events)
    }
}

with open("seal_judge_results.json", "w") as f:
    json.dump(output, f, indent=2)

# Download from Colab
from google.colab import files
files.download("seal_judge_results.json")
print("Results saved and downloaded ✓")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Results saved and downloaded ✓
